# Wiring in Parley & Smoke-Testing the ProofRank Benchmark

This notebook documents, step by step, how the `proofrank` benchmark pipeline was wired up to
run against **MIT Parley** — MIT's OpenAI-compatible LLM gateway (`https://parley.api.mit.edu/v1`)
— and verifies the full `process → solve → postprocess` pipeline end-to-end on two small proof
problems, using Parley's free model, `bedrock/llama-4-maverick-17b` (Llama 4 Maverick 17B Instruct).

Branch: `parley-integration` on this fork (`youssef-chaabouni/proofrank`, forked from
`insait-institute/proofrank`).

Every cell in this notebook was actually executed against the live Parley API — the outputs below
are real, not illustrative.

## 1. Adding Parley as a backend in `src/proofrank/api.py`

`APIQuery` picks its backend via an `api=` string, resolved in `initialize_api_keys`. Most
OpenAI-compatible providers (`xai`, `deepseek`, `glm`, ...) just need an API key env var and a
`base_url`, then fall through to the standard OpenAI client path. Parley fits the same pattern:

```python
elif self.api == "parley":
    self.api_key = os.getenv("PARLEY_API_KEY")
    self.base_url = "https://parley.api.mit.edu/v1"
    self.api = "openai"
```

That's the entire integration surface — `PARLEY_API_KEY` in the environment, model IDs passed as
`"<provider>/<model>"` (e.g. `bedrock/llama-4-maverick-17b`), everything else (retries,
concurrency, tool calling, cost tracking) is shared with every other OpenAI-compatible backend.

One additional fix was required in the shared code path: the default (no-tools) chat-completions
call was sending `tools: null` explicitly in the request body. Parley's request validation rejects
that with `400 "tools must be an array"`, whereas other providers silently tolerate `null`. Fixed
by omitting the `tools` key entirely instead of passing `None`.

## 2. Loading the API key

The key lives in `~/Desktop/code_mit_parley_api/.env` (a separate, local-only project mirroring
Parley's docs) as `PARLEY_API_KEY` — kept out of this repo entirely.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.expanduser("~/Desktop/code_mit_parley_api/.env"))
assert "PARLEY_API_KEY" in os.environ, "PARLEY_API_KEY not found -- check the .env path above"
print("PARLEY_API_KEY loaded:", os.environ["PARLEY_API_KEY"][:14] + "...")

PARLEY_API_KEY loaded: sk-parley-v1-y...


## 3. Direct sanity check of the new `api="parley"` backend

Before touching the benchmark pipeline, a direct call through `APIQuery` confirms the backend
itself works.

In [2]:
from proofrank.api import APIQuery

query = APIQuery(model="bedrock/llama-4-maverick-17b", api="parley", max_tokens=100)
for idx, output, cost in query.run_queries(["Say hello in exactly 5 words."]):
    print("Output:", output)
    print("Cost:  ", cost)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


2026-09-17 15:12:56.054 | INFO     | proofrank.api:run_queries:430 - Running 1 queries.


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.03s/it]

100%|██████████| 1/1 [00:02<00:00,  2.03s/it]

Output: [{'role': 'assistant', 'content': 'Hello, it is nice meeting.'}]
Cost:   {'cost': 5.1e-05, 'input_tokens': 43, 'output_tokens': 8}


## 4. Pre-existing bugs fixed along the way

Getting the benchmark scripts to run at all — on *any* backend, not just Parley — required fixing
a few unrelated, pre-existing issues in the repo:

- **`llmteach` → `proofrank` leftover renames.** `scripts/process.py`, `scripts/solve.py`,
  `scripts/run.py`, `scripts/postprocess.py`, `scripts/recover.py`, and `src/proofrank/parser.py`
  all imported from a `llmteach` package that no longer exists (the project was renamed to
  `proofrank`). Fixed by updating the imports.
- **Missing `proofrank.dag` / `proofrank.dag_classifier` modules.** `results_processors.py`
  imported `SolutionGraphParser` and `GraphMetricsParser` from modules that don't exist anywhere
  in the repo, which crashed the import of `results_processors.py` (and therefore `runner.py` and
  `scripts/run.py`) unconditionally. Since both classes are only actually used inside
  `DAGProcessor.__init__` / `GraphMetricsProcessor.__init__`, the imports were made lazy — the
  same pattern the codebase already uses for the optional `vllm` import in `api.py` — so importing
  the module works, and only instantiating those two specific processors fails.
- **`filelock`'s `unlink_on_release` kwarg.** `src/proofrank/solve.py` constructed a `FileLock`
  with `unlink_on_release=True`, which the resolvable `filelock` versions here don't support
  (`TypeError: UnixFileLock does not support non-default lock options`). Dropped the kwarg — the
  only effect is that per-problem `.lock` files aren't auto-deleted after release, which is
  harmless.
- **Missing `transformers` dependency.** `api.py` unconditionally does
  `from transformers import AutoTokenizer` (used only by the vLLM code path), but `transformers`
  isn't declared in `pyproject.toml`. Installed it manually to unblock any import of `api.py`.
- **README drift.** The README says to run the solver with `scripts/run.py --project <project>`,
  but that's stale: `scripts/run.py` is actually the judge/processor runner
  (`--checker_configs`/`--processor_config`), and the solver (`--project`, `--synchronous`) lives
  in `scripts/solve.py`. This notebook uses the actual, working script names.

## 5. Benchmark config for this smoke test

Three config files plus a two-problem raw dataset were added to drive the benchmark through
Parley:

- `configs/models/parley/llama-4-maverick.yaml` — the model config (`api: parley`,
  `model: bedrock/llama-4-maverick-17b`).
- `configs/projects/parley_smoke_test.yaml` — the project config, pointing at that model.
- `configs/solvers/parley_smoke_test.yaml` — the solver config (prompt template, 1 attempt,
  1 solution per problem — enough for a smoke test, not a real eval run).
- `data/raw/parley_smoke_test/sample.json` — two small proof problems.

In [3]:
import yaml, json

for path in [
    "configs/models/parley/llama-4-maverick.yaml",
    "configs/projects/parley_smoke_test.yaml",
    "configs/solvers/parley_smoke_test.yaml",
]:
    print(f"--- {path} ---")
    print(open(f"../{path}").read())

print("--- data/raw/parley_smoke_test/sample.json ---")
print(json.dumps(json.load(open("../data/raw/parley_smoke_test/sample.json")), indent=2))

--- configs/models/parley/llama-4-maverick.yaml ---
model: bedrock/llama-4-maverick-17b
api: parley
max_tokens: 4096
read_cost: 0.02
write_cost: 0.02
concurrent_requests: 4
temperature: null
top_p: null
human_readable_id: Llama 4 Maverick 17B (Parley, free)
date: "2026-09-17"

--- configs/projects/parley_smoke_test.yaml ---
name: "Parley Smoke Test"
slug: "parley_smoke_test"
admins:
  - "parley-1"

model_configs:
  - "parley/llama-4-maverick"

--- configs/solvers/parley_smoke_test.yaml ---
prompt: |
  Your task is to write a proof solution to the following problem, focusing on accuracy, thoroughness, and clarity. When you write your proof, follow these guidelines:

  - You are creating a proof, not a proof outline. Each step should be carefully explained and documented. If not properly explained, the judge will assume that you cannot explain it, and therefore decrease your grade.
  - You can use general theorems and lemmas, but only if they are well-known.
  - Do not skip computation s

## 6. Step 1/3 — `process.py`: turn raw problems into per-problem files

Reads `data/raw/<project>/sample.json` and fans it out into one JSON file per problem under
`data/unsolved/<project>/`, adding the grading scheme metadata the pipeline expects.

The reset below (removing any previous `unsolved`/`solved`/`postprocess` output for this project)
just makes this notebook idempotent on re-runs — it's not a normal pipeline step.

In [4]:
!rm -rf ../data/unsolved/parley_smoke_test ../data/solved/parley_smoke_test ../data/postprocess/parley_smoke_test
!cd .. && python scripts/process.py --project parley_smoke_test
!find ../data/unsolved/parley_smoke_test -type f

../data/unsolved/parley_smoke_test/smoke/1.json
../data/unsolved/parley_smoke_test/smoke/2.json


## 7. Step 2/3 — `solve.py`: generate proofs via Parley (Llama 4 Maverick)

This is the step that actually calls the Parley API — once per problem, through the
`parley/llama-4-maverick` model config from section 5.

In [5]:
!cd .. && python scripts/solve.py --project parley_smoke_test --synchronous

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Found 2 files to run.
Running 2 problems for project parley_smoke_test
2026-09-17 15:13:05.825 | INFO     | proofrank.solve:solve:32 - Initializing APIQuery for parley/llama-4-maverick...
2026-09-17 15:13:05.826 | INFO     | proofrank.api:run_queries:430 - Running 2 queries.
  0%|                                                    | 0/2 [00:00<?, ?it/s]

 50%|██████████████████████                      | 1/2 [00:04<00:04,  4.41s/it]

100%|████████████████████████████████████████████| 2/2 [00:05<00:00,  2.57s/it]
2026-09-17 15:13:10.994 | INFO     | proofrank.solve:single_loop:111 - Total cost for parley/llama-4-maverick: 3.194e-05
Total cost: 0.00 USD


### Inspecting one of the generated proofs

In [6]:
import json

solved = json.load(open("../data/solved/parley_smoke_test/smoke/1.json"))
attempt = solved["attempts"][0]
print("Problem:", solved["problem"])
print()
print("Model:", attempt["model_id"])
print("Cost:  ", attempt["cost"])
print()
print(attempt["solution"][0]["content"])

Problem: Prove that for every positive integer n, 1 + 2 + ... + n = n(n+1)/2.

Model: parley/llama-4-maverick
Cost:   {'cost': 1.806e-05, 'input_tokens': 199, 'output_tokens': 704}

## Step 1: Understand the problem statement
The problem asks us to prove that the sum of the first n positive integers is equal to n(n+1)/2 for every positive integer n. This is a well-known formula that we need to validate through a rigorous proof.

## Step 2: Choose a method of proof
To prove this statement, we can use mathematical induction, which is a suitable method for proving statements that are true for all positive integers. The principle of mathematical induction involves two main steps: proving the statement is true for n = 1 (base case), and then proving that if it is true for some arbitrary positive integer k, it is also true for k + 1 (inductive step).

## Step 3: Prove the base case (n = 1)
For n = 1, the left-hand side of the equation is 1, and the right-hand side is 1(1+1)/2 = 1. Since both

## 8. Step 3/3 — `postprocess.py`: flatten into a final samples file

Collects every solved problem's attempts across `data/solved/<project>/` into a single flat
`data/postprocess/<project>/test_samples.json` — the format the downstream judge/analysis scripts
consume.

In [7]:
!cd .. && python scripts/postprocess.py --project parley_smoke_test

100%|██████████████████████████████████████████| 2/2 [00:00<00:00, 2714.76it/s]
Total samples: 2
Test samples: 2
Model counts:
parley/llama-4-maverick: 2


In [8]:
import json

samples = json.load(open("../data/postprocess/parley_smoke_test/test_samples.json"))
print(f"Total samples: {len(samples)}")
for s in samples:
    print(f"- {s['problem_id']}: {s['model_id']}, {len(s['solution'])} chars, "
          f"cost=${s['cost']['cost']:.6f}")

Total samples: 2
- smoke_1: parley/llama-4-maverick, 2350 chars, cost=$0.000018
- smoke_2: parley/llama-4-maverick, 1813 chars, cost=$0.000014


## Summary

- Both smoke-test problems were solved end-to-end through Parley's free
  `bedrock/llama-4-maverick-17b` model, at a combined cost of roughly $0.00003.
- The full `process → solve → postprocess` pipeline runs cleanly against the new `"parley"`
  backend, with no Parley-specific code anywhere outside `api.py`'s `initialize_api_keys`.
- Several unrelated, pre-existing bugs (section 4) had to be fixed to get *any* backend running
  through these scripts — worth upstreaming independently of the Parley work.

**Not yet verified** (see `CLAUDE.md` for the up-to-date list): the tool-calling and
`openai_responses` code paths against Parley specifically, and whether Parley's other models
(Claude/GPT-5.x/Gemini) work through this same `"parley"` api option or need their own handling.